# fpl_v2 driver
Thin notebook over the package. All logic lives in the modules; this just runs and inspects.

Run jupyter from the repo root so `import fpl_v2` resolves (`uv run jupyter lab`).

In [1]:
# Ensure the repo root (the dir containing fpl_v2/) is importable, whatever the launch dir.
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / 'fpl_v2').is_dir() and root != root.parent:
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [2]:
from fpl_v2 import pipeline, config

forecast, squad = pipeline.run(save=True)  # refresh=True to re-pull live data
forecast.shape

(425, 18)

In [3]:
cols = ['web_name', 'team_name', 'position', 'cost', 'xG', 'xAG', 'xClean', 'xPoints']
forecast.sort_values('xPoints', ascending=False)[cols].head(30)

,web_name,team_name,position,cost,xG,xAG,xClean,xPoints
294,Haaland,Man City,FWD,155,25.50,2.67,11.180432,110.010000
309,B.Fernandes,Man Utd,MID,120,10.79,12.28,10.119136,99.858758
94,Thiago,Brentford,FWD,80,20.60,1.83,8.020974,87.890000
140,Enzo,Chelsea,MID,70,11.26,7.26,8.232687,85.576078
310,Mbeumo,Man Utd,MID,80,11.98,4.99,10.119136,82.595457
270,O'Reilly,Man City,DEF,65,6.12,2.67,11.180432,79.291265
1,J.Timber,Arsenal,DEF,65,4.71,1.53,15.890859,78.422382
280,Semenyo,Man City,MID,85,11.10,3.12,11.180432,75.321223
0,Gabriel,Arsenal,DEF,80,2.94,1.75,15.890859,74.000951
83,Schade,Brentford,MID,60,12.08,2.05,8.020974,72.985542


In [4]:
print(f"{squad.formation}  |  £{squad.total_cost/10:.1f}m  |  total xP (capt x2): {squad.total_xpoints:.1f}")
print('Captain:', squad.captain['web_name'])
squad.players.sort_values(['position', 'xPoints'], ascending=[True, False])[cols]

4-4-2  |  £78.5m  |  total xP (capt x2): 923.4
Captain: Haaland


,web_name,team_name,position,cost,xG,xAG,xClean,xPoints
270,O'Reilly,Man City,DEF,65,6.12,2.67,11.180432,79.291265
1,J.Timber,Arsenal,DEF,65,4.71,1.53,15.890859,78.422382
271,Guéhi,Man City,DEF,60,4.05,2.37,11.180432,72.601065
325,Thiaw,Newcastle,DEF,50,4.80,0.96,8.410057,60.825028
294,Haaland,Man City,FWD,155,25.50,2.67,11.180432,110.010000
94,Thiago,Brentford,FWD,80,20.60,1.83,8.020974,87.890000
309,B.Fernandes,Man Utd,MID,120,10.79,12.28,10.119136,99.858758
140,Enzo,Chelsea,MID,70,11.26,7.26,8.232687,85.576078
83,Schade,Brentford,MID,60,12.08,2.05,8.020974,72.985542
59,Tavernier,Bournemouth,MID,60,9.06,4.60,8.530422,65.914360


## Tweaking
- Scoring weights: `config.POINTS_FOR_GOAL`, `POINTS_FOR_ASSIST`, `POINTS_FOR_CLEAN`
- Formations / budgets: `config.FORMATIONS`
- Multi-season blend: `config.BLEND_WEIGHTS`
- Manual fixes: `overrides.OVERRIDES`

e.g. re-run a single formation:

In [5]:
from fpl_v2 import optimize

alt = optimize.optimize(forecast, formations={'3-4-3': config.FORMATIONS['3-4-3']})
alt.players.sort_values('xPoints', ascending=False)[cols]

,web_name,team_name,position,cost,xG,xAG,xClean,xPoints
294,Haaland,Man City,FWD,155,25.50,2.67,11.180432,110.010000
309,B.Fernandes,Man Utd,MID,120,10.79,12.28,10.119136,99.858758
94,Thiago,Brentford,FWD,80,20.60,1.83,8.020974,87.890000
140,Enzo,Chelsea,MID,70,11.26,7.26,8.232687,85.576078
270,O'Reilly,Man City,DEF,65,6.12,2.67,11.180432,79.291265
1,J.Timber,Arsenal,DEF,65,4.71,1.53,15.890859,78.422382
83,Schade,Brentford,MID,60,12.08,2.05,8.020974,72.985542
59,Tavernier,Bournemouth,MID,60,9.06,4.60,8.530422,65.914360
237,Calvert-Lewin,Leeds,FWD,60,15.57,1.06,8.629769,65.460000
325,Thiaw,Newcastle,DEF,50,4.80,0.96,8.410057,60.825028
